# 4.1 Where Chains Break (Why Graphs)

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Learning objectives

- Explain what a chain is (data flows one way, through fixed steps) and where that shape runs out of room.
- Name the four breaks: loops, branching on state, shared state that many steps update, and pausing mid-run.
- See the hand-rolled ReAct loop from lesson 2.3 as a graph in disguise, and say what making it a real graph buys you.
- Decide, for a real task, whether a chain is enough or a graph is warranted.

## Chains are straight lines

Everything in Modules 0 to 3 was built on one shape: the chain. A prompt goes in, a model runs, a parser reads the output, and data flows forward, one step after the next. Even the fancier plumbing (`RunnableParallel`, `RunnableBranch`) keeps that character: the route through the steps is fixed at build time, and each step passes its result to the next step.

That shape is fine, and often the right choice. A summarise-then-translate pipeline, a classify-then-format flow, a RAG answer: fixed steps, one direction, done. But four common requirements do not fit a straight line, and when they appear, people start bending the chain with workarounds until the code is harder to read than the problem. Each workaround is a graph in disguise. This module teaches you to write the graph directly.

## Break 1: loops

You already met the first break. ReAct is a loop: thought, action, observation, repeat until the model stops calling tools. In lesson 2.3 you wrote it by hand:

In [ ]:
for step in range(6):
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)
    if not ai_msg.tool_calls:
        break
    for call in ai_msg.tool_calls:
        messages.append(by_name[call["name"]].invoke(call))

Draw that as a diagram and it is a graph: `agent -> tools -> agent -> tools -> ... -> END`. The loop cap, the exit condition, the tool dispatch: all control flow that LCEL never saw. The chain abstraction ended at `bind_tools`; the loop around it was yours, invisible to every framework feature. Retries, streaming, persistence, a budget: you had to wire each one by hand, in the same `for` loop as your business logic.

Lesson 2.3 also told you that `create_agent` already runs on the LangGraph runtime. That was the reveal: the "agent" you built with one function call is a compiled graph of two nodes, `agent` and `tools`, with a conditional edge looping between them. This module teaches the runtime directly, so when the standard loop does not fit, you can build your own graph instead of hand-rolling another `for` loop.

## Break 2: branching on state

Suppose you run a support inbox. Every message gets classified, then goes down one of three paths: billing, technical, or nowhere (the run ends). In LCEL you would reach for `RunnableBranch`. It works, once. Then the requirements arrive: the classifier's label should be stored for auditing; the billing path needs the customer's plan tier, which only the classifier knows; and the "nowhere" branch should count how many messages were dropped, so a dashboard can read the counter.

Now the chain is passing one value from step to step, but the control flow needs a shared board that several steps read and write. In a graph, the router is a plain function of state:

In [ ]:
def route(state: SupportState) -> str:
    return state["category"]     # "billing", "technical", or "none"

builder.add_conditional_edges("classify", route, {
    "billing": "billing_node",
    "technical": "technical_node",
    "none": END,
})

Two things to notice. First, the model never picks the destination: the router does, deterministically, from state. Control flow stays in your code, not in the model's output. Second, the router is a pure function with no LLM in it, which means you can unit test it with a dictionary, no API key, no network. Routing is where agent bugs live; this is the one part you can test properly.

## Break 3: shared state that many steps update

A chain passes one value between steps. Real workflows need a board: the current plan, a list of findings from parallel research workers, a retry counter, the draft answer. When you need that in LCEL, you end up threading dictionaries through every step and merging them by hand, which is state management with no help from the framework.

Graphs name the board explicitly: state. Every node reads the whole board and returns only what changed. When two nodes run in parallel and both append to the findings list, a reducer decides how their updates merge (append, not overwrite), so branches cannot clobber each other. Sketch of the idea; the mechanics are lesson 4.2:

In [ ]:
class ResearchState(TypedDict):
    question: str
    findings: Annotated[list, operator.add]   # parallel workers append here

That `Annotated` annotation is the difference between "each worker adds its findings" and "the last worker wins". The exact bug you would otherwise spend a day on in hand-rolled async code.

## Break 4: pausing mid-run

A chain runs to completion. But some steps should not run on their own: sending an invoice email, deleting customer data, placing an order, or any write that a wrong model output could make expensive. You want the run to stop, a human to approve or edit, and the run to continue.

Hand-rolling that pause means persisting the whole state yourself: the message history, the loop position, the partial results. Serialise it, store it, rebuild it on resume, hope nothing drifted. In a graph, pausing is a primitive: a checkpointer saves the state after every step, and an `interrupt()` call freezes the run until a person answers. Sketch; lesson 4.4 builds it properly:

In [ ]:
def approve(state: State):
    decision = interrupt({"question": "Send this invoice email?", "draft": state["draft"]})
    return {"approved": decision == "yes"}

The reason this works is the same shape: the graph's state after each node is a complete snapshot of the run, so resuming is just loading a snapshot. A chain cannot do this because it never names its intermediate state at all.

## When to stay with a chain

Graphs are not an upgrade. They are a different shape, with real costs: more code, more concepts, more places to get the wiring wrong. Stay with a chain when the flow is fixed, each step has one output, and no step needs to know what the others did. A surprising amount of production work is exactly that, and a two-step chain wearing a graph costume is just harder to read.

Switch when one of the four breaks shows up: a loop whose exit is decided at runtime, routing on shared state, many steps updating one board, or a pause for a human. And remember the rule from lesson 2.3: start with `create_agent`, and build a custom graph only when you can point at a specific behaviour the harness cannot express. The ReAct loop covers most real agents; the rest of this module is for the ones it does not.

## What can go wrong

- **Graph-shaped thinking applied to chain-shaped problems.** Two fixed steps do not need nodes, edges, and state. Over-engineering is the most common LangGraph mistake in the wild, and it is made by reaching for the graph before the problem demands it.
- **Loops without budgets.** A cycle in a graph is an infinite loop wearing nicer clothes. LangGraph's default `recursion_limit` (25) is the hard backstop; it raises `GraphRecursionError` when a run exceeds it. Set a real budget (a retry counter in state, a model-call limit) instead of relying on the backstop, because every iteration before the limit is money spent.
- **The model deciding control flow.** Letting the model route by calling a "go to billing" tool works until the model invents a destination or misroutes under pressure. Classify with the model, route with code, and the route map is the complete list of places a run can go.
- **Hand-rolled state dicts between chain steps.** Without a declared schema, a typo in a key silently becomes a new field and the data you meant to pass is never read. Declaring the state schema is what turns that silent bug into a loud one.
- **Parallel branches sharing one field with no reducer.** Two workers both writing a plain field means the last write wins and the first worker's output vanishes. This is the bug reducers exist to prevent; lesson 4.2 shows the fix.

## Check your understanding

**1. A chain passes data one way through fixed steps. Name the four breaks and the graph mechanism that answers each.**
Loops: a cycle of edges (agent -> tools -> agent) instead of a hand-rolled `for` loop. Branching: conditional edges with a router function of state. Shared state: an explicit state schema that every node reads and returns updates to, with reducers merging parallel writes. Pausing: interrupts plus checkpointed snapshots, so a run can freeze mid-flow and resume later.

**2. The hand-rolled ReAct loop from lesson 2.3 is "a graph in disguise". Explain what that means, and what becomes visible when you build it as a real graph.**
Drawn as a diagram, the loop is already nodes (agent, tools) and edges (including the back-edge from tools to agent) with an exit condition. As a hand-rolled loop, that control flow is invisible to the framework: retries, streaming, persistence, and budgets all have to be wired by hand. As a compiled graph, the same shape gets those features from the runtime, and the control flow can be inspected, streamed, and debugged instead of hidden inside your `for` loop.

**3. In the support-inbox example, why should the model classify but not route?**
The model classifies because classification is a judgement call over messy text, which is what models are for. The router is deterministic code reading `state["category"]`, so the set of possible destinations is exactly the route map and nothing else. If the model routes directly (for example by calling a routing tool), it can invent destinations, and the routing logic cannot be unit tested without the model. Keep the judgement in the model and the control flow in your code.

**4. Two research workers run in parallel and both append findings to a shared list. What goes wrong without a reducer, and what does the `Annotated[list, operator.add]` annotation change?**
Without a reducer, a node's returned value replaces the field, so the second worker's findings overwrite the first worker's: last write wins and results vanish. With the `operator.add` reducer, LangGraph appends each node's returned list to the existing one instead of replacing it, so parallel branches merge instead of clobbering each other.

**5. A colleague proposes rebuilding a working two-step summarise-then-translate chain as a LangGraph graph "for future flexibility". What do you say?**
Say no, or at least not yet. The flow is fixed, each step has one output, no step needs the others' state, and there is no loop, branch, pause, or shared board. That is a chain-shaped problem; a graph adds code and concepts with no payoff. Revisit only when one of the four breaks actually appears, because you can point at a specific behaviour the chain cannot express. YAGNI applies to orchestration frameworks too.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.